## 1. Instalando dependências

In [ ]:
!pip install --upgrade setuptools
!pip install git+https://github.com/IAZero/wisardpkg.git@develop
!pip install ipykernel
!pip install ipywidgets
!pip install matplotlib
!pip install numba
!pip install numpy
!pip install pybind11
!pip install scikit-learn

!pip install tqdm

  Cloning https://github.com/IAZero/wisardpkg.git (to revision develop) to /tmp/pip-req-build-1tlkofn1
  Running command git clone --filter=blob:none --quiet https://github.com/IAZero/wisardpkg.git /tmp/pip-req-build-1tlkofn1
  Running command git checkout -b develop --track origin/develop
  Switched to a new branch 'develop'
  Branch 'develop' set up to track remote branch 'develop' from 'origin'.
  Resolved https://github.com/IAZero/wisardpkg.git to commit c51b395c0e85ae007bb7f8699515a3136356ef65
  Preparing metadata (setup.py) ... done


In [ ]:
!git clone https://gist.github.com/1fe5c78be9a5ec2d507e8118e2c99740.git
!mv 1fe5c78be9a5ec2d507e8118e2c99740/*.py ./
!rm -rf ./1fe5c78be9a5ec2d507e8118e2c99740

Cloning into '1fe5c78be9a5ec2d507e8118e2c99740'...
remote: Enumerating objects: 5, done.
remote: Total 5 (delta 0), reused 0 (delta 0), pack-reused 5 (from 1)
Receiving objects: 100% (5/5), 4.58 KiB | 4.58 MiB/s, done.


## 2. Importando dependências

In [ ]:
# medicao de tempo de execucao de treinos e predicoes
import time

# barra de progresso para for loop da simulacao
from tqdm import tqdm

# algebra linear e manipulacao de dados
import numpy as np
import pandas as pd
from scipy.stats import norm

# modelo wisard de machine learning (wnn)
import wisardpkg as wp

# visualizacao de dados
import matplotlib.pyplot as plt
import seaborn as sns

# encoders para binarizacao de dados
from encoders import (
    ThermometerEncoder,
    CircularThermometerEncoder,
    FloatBinaryEncoder,
    process_input,
    Morph,
    CodeWord,
    NeighborsSuffix,
    MockNeighborsSuffix
    )

# download, pre-processamento e metricas de validacao
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# avaliacao de tipos de atributos
from pandas.core.dtypes.api import is_numeric_dtype

# componentes visuais
from ipywidgets import fixed, interact_manual
from IPython.display import display

plt.rcParams['figure.figsize'] = [6, 6]
plt.rcParams['figure.dpi'] = 80


Configurando função para termômetro linear a partir de um *dataset* fornecido.

In [ ]:
# para um conjunto de dados X fornecido
def thermometerEncoder(X, thersize) : # X é um conjunto de dados
  if X.empty or X.isnull().all():
      print(f"Aviso: serie especificada para o thermometerEncoder esta vazia ou repleta de NaNs. Retornando lista vazia.")
      return []

  min_val = X.min()
  max_val = X.max()
  if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
      if pd.isna(min_val) or pd.isna(max_val):
          min_val = X.iloc[0] - 0.5 if X.iloc[0] is not None else 0
          max_val = X.iloc[0] + 0.5 if X.iloc[0] is not None else 1
          if min_val == max_val: # Still problematic if series was constant 0
              min_val = 0
              max_val = 1 # Fallback to a default range

  thermometer = ThermometerEncoder(minimum=min_val, maximum=max_val, resolution=thersize)
  X_bin = [thermometer.encode(x) for x in X.dropna()] # somente valores nao nulos (NaN)
  return X_bin

Configurando função para termômetro gaussiano a partir de um *dataset* fornecido.

In [ ]:
def gaussianThermometer(X, thersize, mean=0, std=1):
  X = X.fillna(mean)

  thresholds = np.array([
        mean + std * norm.ppf(
            (i + 1)/(thersize + 1)
        )
        for i in range(thersize)
    ])

  encoded = (
      X.to_numpy().reshape(-1, 1)
      >= thresholds
  ).astype(np.uint8)

  return encoded

## 3. Etapa de pré-processamento e preparação (ETL — extract, transform, load) para os dados do problema

In [ ]:
def pre_processing(path, thersize, target=['label', 'type']) :
  file_path = '/content/Train_Test_IoT_' + path + '.csv'
  df = pd.read_csv(file_path)

  print(f"\n--- Pre-processamento de {path} com thersize={thersize} ---\n")
  print("DataFrame original:")
  for col, typ in df.dtypes.items() :
    print(f"{col} : {typ}")
  print()

  # remover colunas nao informativas
  to_drop = ['date', 'time']
  df = df.drop(columns=to_drop, errors='ignore')

  # separar o alvo dos atributos
  labels = []

  if target[1] in df.columns :
    print(f"Utilizando {target[1]} como rotulo alvo.\n")
    le = LabelEncoder()
    labels = [str(l) for l in le.fit_transform(df[target[1]])]
    features_df = df.drop(columns=target, errors='ignore')
    print(f"Rotulos {target[1]} codificados.\n")

  # cai de volta em outro rotulo se o desejado nao for encontrado
  elif target[0] in df.columns :
    print(f"Aviso: {target[1]} nao encontrado no dataset. Voltando para a coluna {target[0]} original.")
    labels = [str(l) for l in df[target[0]].values]
    features_df = df.drop(columns=[target[0]])

  else :
    print(f"Aviso: Nenhuma das colunas {target[1]} ou {target[0]} foram encontradas para serem alvo.")
    features_df = df.copy()


  processed_feature_bits_per_column = []

  for col in features_df.columns :
    feature_series = features_df[col]

    if pd.api.types.is_numeric_dtype(feature_series) :
      print(f"Processando coluna numerica '{col}' com thermometerEncoder (thersize={thersize})")
      # print(f"Processando coluna numerica '{col}' com gaussianThermometer (thersize={thersize}, \u03bc =0, \u03c3 =1)")

      try :
        encoded_binary_list = thermometerEncoder(feature_series, thersize)
        # encoded_binary_list = gaussianThermometer(feature_series, thersize)
        # if encoded_binary_list :
        if len(encoded_binary_list) > 0 :
          processed_feature_bits_per_column.append(encoded_binary_list)
          print(f"Coluna {col} binarizada.\n")
      except Exception as e :
        print(f"Erro ao binarizar a coluna numerica '{col}': {e}")

    elif pd.api.types.is_string_dtype(feature_series):
      print(f"Processando coluna categorica '{col}' com LabelEncoder() e thermometerEncoder(thersize={thersize})")
      # print(f"Processando coluna categorica '{col}' com LabelEncoder() e gaussianThermometer (thersize={thersize}, \u03bc =0, \u03c3 =1)")

      try :
        le = LabelEncoder()
        numeric_representation = le.fit_transform(feature_series.fillna(''))
        encoded_binary_list = thermometerEncoder(pd.Series(numeric_representation), thersize)
        # encoded_binary_list = gaussianThermometer(pd.Series(numeric_representation), thersize)

        # if encoded_binary_list :
        if len(encoded_binary_list) > 0 :
          processed_feature_bits_per_column.append(encoded_binary_list)
          print(f"Coluna {col} binarizada.\n")
      except Exception as e :
        print(f"Erro ao binarizar a coluna categorica '{col}': {e}")

    else :
      print(f"Pulando coluna '{col}' de tipo nao suportado: {feature_series.dtype}")

  combined_binary_features_per_sample = []

  if processed_feature_bits_per_column :
    num_samples = len(processed_feature_bits_per_column[0])

    for i in range(num_samples) :
      current_sample_combined_bits = []
      for feature_bit_list in processed_feature_bits_per_column :
        if i < len(feature_bit_list) :
          current_sample_combined_bits.extend([int(bit) for bit in feature_bit_list[i]])
        else :
          print(f"Aviso: Lista de features tem menos amostras do que o esperado no indice {i}.")

      combined_binary_features_per_sample.append(current_sample_combined_bits)
  else :
    print(f"Nenhuma feature foi processada com sucesso para binarizacao.")

  # criando o WiSARD DataSet
  if combined_binary_features_per_sample and labels and len(combined_binary_features_per_sample) == len(labels):
        print(f"Dataset processado com sucesso com {len(combined_binary_features_per_sample)} amostras.")
        return combined_binary_features_per_sample, labels
  else:
      print("Nao conseguiu criar os dados binarizados: Tamanhos diferentes ou vazios.")
      return None, None

## 4. Treinamento do modelo WiSARD (Wilkes, Stonham and Aleksander Recognition Device)

Será realizado um treinamento com $20\%$ dos dados dedicados para teste. Além disso, métricas importantes como  **acurácia**, **precisão**, **recall** e **F1-score** também serão obtidas para cada um dos conjuntos de dados (*Fridge, Garage Door, GPS Tracker, Modbus, Motion Light, Thermostat, Weather*).

Realizando `n` simulações para o modelo construído, variando os hiperparâmetros `addressSize`, `thersize` e *dataset* escolhido:

In [ ]:
def analytics(datasets, thermometer_values, address_size_values, sims=100) :
  results = []

  for d in datasets :
    for t in thermometer_values :

      try :
        X, y = pre_processing(path=d, thersize=t)

        if X is None or y is None:
          continue
      except Exception as e:
        print(f"Erro no preprocessamento do dataset {d} com thermometer={t} : {e}")
        continue

      for a in address_size_values :
        acc, prec, rec, f1 = [], [], [], []
        training_time, prediction_time = [], []

        print(f"\nSimulando WiSARD {sims} vezes para thersize={t} e addressSize={a}...")
        print(f"Dataset: {d}")

        print("--- Convertendo dados para o formato interno do WiSARD ---")


        for s in tqdm(range(sims), desc='Classificando os dados do teste') :
          X_train, X_test, y_train, y_test = train_test_split(
          X, y,
          test_size=0.2,
          # random_state=s,
          stratify=y
          )

          train_dataset = wp.DataSet()
          for x, y_val in zip(X_train, y_train):
              train_dataset.add(x, str(y_val))

          test_dataset = wp.DataSet()
          for x in X_test:
              test_dataset.add(x)


          # Instanciação e Treinamento
          model = wp.Wisard(a)

          start_train = time.time()
          model.train(train_dataset)
          training_time.append(time.time() - start_train)
          # print(f"Tempo de treinamento: {training_time:.2f} segundos")

          # print(f"\n--- Classificando os dados do teste ---")
          start_pred = time.time()
          y_pred_str = model.classify(test_dataset)
          prediction_time.append(time.time() - start_pred)

          y_type = type(y_test[0]) if len(y_test) > 0 else int
          y_pred = [y_type(val) for val in y_pred_str]
          # print(f"Tempo de teste: {prediction_time:.2f} segundos")

          # Metricas
          acc.append(accuracy_score(y_test, y_pred))
          prec.append(precision_score(y_test, y_pred, average='macro', zero_division=0))
          rec.append(recall_score(y_test, y_pred, average='macro', zero_division=0))
          f1.append(f1_score(y_test, y_pred, average='macro', zero_division=0))

        print(f"Amostras de treinamento: {len(X_train)}")
        print(f"Amostras de teste: {len(X_test)}\n")

        # salvando as estatísticas
        results.append({
            "Dataset": d,
            "t": t,
            "a": a,
            "Acuracia (media)": round(float(np.mean(acc)), 3),
            "Precisao (media)": round(float(np.mean(prec)), 3),
            "Recall (media)": round(float(np.mean(rec)), 3),
            "F1-Score (media)": round(float(np.mean(f1)), 3),
            "Tempo Treino (s)": round(float(np.mean(training_time)), 3),
            "Tempo Teste (s)": round(float(np.mean(prediction_time)), 3)
        })

        print(f"\nResultados para dataset {d}, thersize={t} e addressSize={a}:")
        print(f"Acuracia (media): {round(float(np.mean(acc)), 3)}")
        print(f"Precisao (media): {round(float(np.mean(prec)), 3)}")
        print(f"Recall (media): {round(float(np.mean(rec)), 3)}")
        print(f"F1-Score (media): {round(float(np.mean(f1)), 3)}")
        print(f"Tempo de treinamento (s): {round(float(np.mean(training_time)), 3)}")
        print(f"Tempo de teste (s): {round(float(np.mean(prediction_time)), 3)}\n")

  return results

Definindo os parâmetros dos modelos:


*   Bits do termômetro: $\{4, 8, 16, 32\}$ bits;
*   Tamanhos de endereçamento do modelo: $\{4, 8, 16\}$ bits;
*   *Datasets* mencionados anteriormente.



In [ ]:

datasets = [
    'Fridge',
    'Garage_Door',
    'GPS_Tracker',
    'Modbus',
    'Motion_Light',
    'Thermostat',
    'Weather'
]

thermometer_values = [4, 8, 16, 32]
address_size_values = [4, 8, 16]



Colhendo as métricas desejadas (as de melhor acerto):

In [ ]:
def the_best(results) :
  bests = []
  df = pd.DataFrame(results)

  for d in datasets :

    temp_df = df[df["Dataset"] == d]

    if not df.empty:
        # Ordena de forma decrescente pelas médias de Acurácia e Precisão
        temp_df = temp_df.sort_values(
            by=["Acuracia (media)", "Precisao (media)"],
            ascending=[False, False]
        )
        bests.append(temp_df.index[0])

  df = df.loc[bests].reset_index(drop=True)

  return df



In [ ]:
def generate_results(datasets=datasets, thermometer_values=thermometer_values, address_size_values=address_size_values) :
  try :
    results = analytics(datasets, thermometer_values, address_size_values)
    print("Resultados abaixo:")
    print(results)
  except Exception as e:
      print(f"Ocorreu um erro durante a execução da função analytics: {e}")
      import traceback
      traceback.print_exc()

  return results

generate_results(datasets, thermometer_values, address_size_values)


--- Pre-processamento de Fridge com thersize=4 ---

DataFrame original:
date : object
time : object
fridge_temperature : float64
temp_condition : object
label : int64
type : object

Utilizando type como rotulo alvo.

Rotulos type codificados.

Processando coluna numerica 'fridge_temperature' com thermometerEncoder (thersize=4)
Coluna fridge_temperature binarizada.

Processando coluna categorica 'temp_condition' com LabelEncoder() e thermometerEncoder(thersize=4)
Coluna temp_condition binarizada.

Dataset processado com sucesso com 39944 amostras.

Simulando WiSARD 100 vezes para thersize=4 e addressSize=4...
Dataset: Fridge
--- Convertendo dados para o formato interno do WiSARD ---


Classificando os dados do teste: 100%|██████████| 100/100 [00:34<00:00,  2.94it/s]


Amostras de treinamento: 31955
Amostras de teste: 7989


Resultados para dataset Fridge, thersize=4 e addressSize=4:
Acuracia (media): 0.468
Precisao (media): 0.216
Recall (media): 0.263
F1-Score (media): 0.228
Tempo de treinamento (s): 0.006
Tempo de teste (s): 0.041


Simulando WiSARD 100 vezes para thersize=4 e addressSize=8...
Dataset: Fridge
--- Convertendo dados para o formato interno do WiSARD ---


Classificando os dados do teste: 100%|██████████| 100/100 [00:33<00:00,  3.03it/s]


Amostras de treinamento: 31955
Amostras de teste: 7989


Resultados para dataset Fridge, thersize=4 e addressSize=8:
Acuracia (media): 0.427
Precisao (media): 0.155
Recall (media): 0.286
F1-Score (media): 0.164
Tempo de treinamento (s): 0.006
Tempo de teste (s): 0.018


Simulando WiSARD 100 vezes para thersize=4 e addressSize=16...
Dataset: Fridge
--- Convertendo dados para o formato interno do WiSARD ---


...

Classificando os dados do teste: 100%|██████████| 100/100 [00:44<00:00,  2.25it/s]


Amostras de treinamento: 31408
Amostras de teste: 7852


Resultados para dataset Weather, thersize=16 e addressSize=16:
Acuracia (media): 0.641
Precisao (media): 0.681
Recall (media): 0.552
F1-Score (media): 0.592
Tempo de treinamento (s): 0.019
Tempo de teste (s): 0.069


--- Pre-processamento de Weather com thersize=32 ---

DataFrame original:
date : object
time : object
temperature : float64
pressure : float64
humidity : float64
label : int64
type : object

Utilizando type como rotulo alvo.

Rotulos type codificados.

Processando coluna numerica 'temperature' com thermometerEncoder (thersize=32)
Coluna temperature binarizada.

Processando coluna numerica 'pressure' com thermometerEncoder (thersize=32)
Coluna pressure binarizada.

Processando coluna numerica 'humidity' com thermometerEncoder (thersize=32)
Coluna humidity binarizada.

Dataset processado com sucesso com 39260 amostras.

Simulando WiSARD 100 vezes para thersize=32 e addressSize=4...
Dataset: Weather
--- Convertendo dado

Classificando os dados do teste: 100%|██████████| 100/100 [02:51<00:00,  1.72s/it]


Amostras de treinamento: 31408
Amostras de teste: 7852


Resultados para dataset Weather, thersize=32 e addressSize=4:
Acuracia (media): 0.57
Precisao (media): 0.599
Recall (media): 0.415
F1-Score (media): 0.448
Tempo de treinamento (s): 0.039
Tempo de teste (s): 1.276


Simulando WiSARD 100 vezes para thersize=32 e addressSize=8...
Dataset: Weather
--- Convertendo dados para o formato interno do WiSARD ---


Classificando os dados do teste:  81%|████████  | 81/100 [01:04<00:14,  1.34it/s]

In [ ]:
def exhibit_results(results):
    best_df = the_best(results)
    df = pd.DataFrame()
    df['Dataset'] = best_df['Dataset']
    df['thersize'] = best_df['t']
    df['addressSize'] = best_df['a']

    df["Accuracy"] = best_df["Acuracia (media)"].apply(lambda v: f"{v:.3f}")
    df["Precision"] = best_df["Precisao (media)"].apply(lambda v: f"{v:.3f}")
    df["Recall"]   = best_df["Recall (media)"].apply(lambda v: f"{v:.3f}")
    df["F1-Score"] = best_df["F1-Score (media)"].apply(lambda v: f"{v:.3f}")

    df['Training Time (s)'] = best_df['Tempo Treino (s)'].apply(lambda v: f"{v:.3f}")
    df['Predition Time (s)'] = best_df['Tempo Teste (s)'].apply(lambda v: f"{v:.3f}")

    print("\n==================== TABELA FINAL (MELHORES HIPERPARÂMETROS - APENAS MÉDIAS) ====================")
    display(df.style.set_properties(**{'text-align': 'center'}).hide(axis='index'))



exhibit_results(results)


==================== TABELA FINAL (MELHORES HIPERPARÂMETROS - APENAS MÉDIAS) ====================


Dataset,thersize,addressSize,Accuracy,Precision,Recall,F1-Score,Training Time (s),Predition Time (s)
Fridge,32,8,0.500,0.266,0.286,0.269,0.019,0.131
Garage_Door,8,4,0.502,0.174,0.249,0.182,0.009,0.046
GPS_Tracker,32,16,0.528,0.419,0.359,0.357,0.015,0.103
Modbus,32,16,0.465,0.477,0.233,0.240,0.030,0.095
Motion_Light,4,4,0.380,0.047,0.125,0.069,0.009,0.050
Thermostat,8,8,0.456,0.075,0.143,0.090,0.009,0.046
Weather,32,16,0.675,0.720,0.601,0.640,0.028,0.130


### **Trabalhos Futuros**

Os valores obtidos para acurácia, precisão e recall foram considerados um pouco abaixo do esperado. Propostas futuras para aumentarem estes valores incluem:

1. Aumento do tamanho do endereçamento do modelo (`addressSize`) levaria o aumento da precisão e da acurácia (evitam falsos positivos);
2. Diminuição do tamanho do endereçamento do modelo (`addressSize`) levaria o aumento do recall (evitam falsos negativos);
3. Além do `GaussianThermometer`proposto — que apresentou acurácias e precisões muito baixas —, utilizar outros encoders, como:
    - `wp.SimpleThermometer(thermometerSize = 4, minimum = 0, maximum = 10)`
    - `wp.DynamicThermometer(thermometerSizes=thermometerSizes, minimum=minimum, maximum=maximum)`
    - `CircularThermometerEncoder`
4. Por outro lado, a sugestão de utilizar *bleaching* não foi acatada, pois gostaríamos de experimentar as métricas com o WiSARD "mais puro" possível.